# Part A — Problem & Dataset Selection

**Final Project: Philippine Peso Depreciation Early-Warning Model: Predicting the Next-Month PHP/USD Exchange Rate Using Linear Regression and PCA
**

Every good model starts with a clearly defined problem and an honest look at the data before any math happens.

---
**Student name:** Alfajaro, Ghan Vincent | Arile, Charles Jullian | Asuar, Jhoe Maven | Mondragon, Leneth | Ulitin, Justin | **Date:** November 08, 2026

---


**What to do:** Choose an open dataset and a target variable to predict. Load it and report its basic shape and structure.

In [1]:
import numpy as np
import pandas as pd

np.set_printoptions(precision=4, suppress=True)
pd.set_option("display.max_columns", None)

In [2]:
# Read the monthly worksheet directly from the official BSP Excel workbook.
BSP_URL = "https://www.bsp.gov.ph/statistics/external/pesodollar.xlsx"

monthly_raw = pd.read_excel(
    BSP_URL,
    sheet_name="monthly",
    header=4,
    usecols="A:D"
)

print("Columns in the BSP monthly worksheet:")
print(monthly_raw.columns.tolist())
display(monthly_raw.head(15))

# The workbook columns shown by the source are: Period (year),
# Unnamed: 2 (month), and Average (monthly average exchange rate).
year = pd.to_numeric(monthly_raw["Period"], errors="coerce").ffill()
month_name = monthly_raw["Unnamed: 2"].astype(str).str.strip().str.lower()
rate = pd.to_numeric(monthly_raw["Average"], errors="coerce")

month_numbers = {
    "january": 1, "february": 2, "march": 3, "april": 4,
    "may": 5, "june": 6, "july": 7, "august": 8,
    "september": 9, "october": 10, "november": 11, "december": 12
}
month = month_name.map(month_numbers)

# Combine the filled year and month into proper monthly dates.
dates = pd.to_datetime(
    {"year": year, "month": month, "day": 1},
    errors="coerce"
)

monthly = pd.DataFrame({"Date": dates, "Rate": rate}).dropna()
monthly["Date"] = pd.to_datetime(monthly["Date"]).dt.to_period("M").dt.to_timestamp()
monthly = monthly.sort_values("Date").drop_duplicates("Date", keep="last")
monthly = monthly.set_index("Date")["Rate"].astype(float)
monthly.index = pd.DatetimeIndex(monthly.index)
monthly.index.name = "FeatureMonth"

print("\nMonthly series loaded successfully.")
print("First month:", monthly.index.min().strftime("%Y-%m"))
print("Last month:", monthly.index.max().strftime("%Y-%m"))
print("Available monthly observations:", len(monthly))

Columns in the BSP monthly worksheet:
['Unnamed: 0', 'Period', 'Unnamed: 2', 'Average']


,Unnamed: 0,Period,Unnamed: 2,Average
0,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN
2,NaN,1945.0,January,2.0
3,NaN,NaN,February,2.0
4,NaN,NaN,March,2.0
5,NaN,NaN,April,2.0
6,NaN,NaN,May,2.0
7,NaN,NaN,June,2.0
8,NaN,NaN,July,2.0
9,NaN,NaN,August,2.0



Monthly series loaded successfully.
First month: 1945-01
Last month: 2026-10
Available monthly observations: 982


In [3]:
start_date = "1990-01-01"
end_date = "2026-09-01"
expected_months = pd.date_range(start=start_date, end=end_date, freq="MS")

monthly = monthly.loc[start_date:end_date].reindex(expected_months)
monthly.index = pd.DatetimeIndex(monthly.index)
monthly.index.name = "FeatureMonth"

missing_months = monthly[monthly.isna()].index
print("Expected months:", len(expected_months))
print("Missing monthly rates:", len(missing_months))

if len(missing_months) > 0:
    print("Missing months:", missing_months.strftime("%Y-%m").tolist())
    raise ValueError(
        "Some monthly exchange rates are missing. Check the BSP worksheet "
        "before continuing; do not fill missing exchange rates with guesses."
    )

# Construct the three features and the next-month target.
model_data = pd.DataFrame(index=monthly.index)
model_data["CurrentRate"] = monthly
model_data["Trailing3MonthAverage"] = monthly.rolling(window=3).mean()
model_data["YearOverYearChange"] = monthly - monthly.shift(12)
model_data["NextMonthRate"] = monthly.shift(-1)

feature_names = [
    "CurrentRate",
    "Trailing3MonthAverage",
    "YearOverYearChange"
]
target_name = "NextMonthRate"

# Remove rows that do not have all required feature and target values.
model_data = model_data.dropna(subset=feature_names + [target_name])

# Separate the feature matrix (X) and target vector (y).
X_raw = model_data[feature_names].to_numpy(dtype=float)
y = model_data[target_name].to_numpy(dtype=float)

part_a_dataset = model_data.copy()
part_a_dataset["TargetMonth"] = part_a_dataset.index + pd.DateOffset(months=1)

n_samples, n_features = X_raw.shape
print("\nPrepared dataset shape:", part_a_dataset.shape)
print("Number of samples (n):", n_samples)
print("Number of features (p):", n_features)
print("Feature names:", feature_names)
print("Target variable:", target_name)
print("Feature months:", part_a_dataset.index.min().strftime("%Y-%m"),
      "to", part_a_dataset.index.max().strftime("%Y-%m"))
print("Target months:", part_a_dataset["TargetMonth"].min().strftime("%Y-%m"),
      "to", part_a_dataset["TargetMonth"].max().strftime("%Y-%m"))

Expected months: 441
Missing monthly rates: 0

Prepared dataset shape: (428, 5)
Number of samples (n): 428
Number of features (p): 3
Feature names: ['CurrentRate', 'Trailing3MonthAverage', 'YearOverYearChange']
Target variable: NextMonthRate
Feature months: 1991-01 to 2026-08
Target months: 1991-02 to 2026-09


In [4]:
print("First five observations:")
display(part_a_dataset.head())

print("\nDescriptive statistics for the three independent features:")
display(pd.DataFrame(X_raw, columns=feature_names).describe())

print("\nDescriptive statistics for the target:")
display(pd.Series(y, name=target_name).describe().to_frame())

print("\nMissing values in features and target:")
display(part_a_dataset[feature_names + [target_name]].isna().sum().to_frame("Missing Values"))

print("Duplicate feature-month dates:", part_a_dataset.index.duplicated().sum())

First five observations:


,CurrentRate,Trailing3MonthAverage,YearOverYearChange,NextMonthRate,TargetMonth
FeatureMonth,,,,,
1991-01-01,28.000000,28.000000,5.535636,28.000000,1991-02-01
1991-02-01,28.000000,28.000000,5.375550,28.000000,1991-03-01
1991-03-01,28.000000,28.000000,5.240636,27.926143,1991-04-01
1991-04-01,27.926143,27.975381,5.165254,27.818591,1991-05-01
1991-05-01,27.818591,27.914911,4.916773,27.795500,1991-06-01



Descriptive statistics for the three independent features:


,CurrentRate,Trailing3MonthAverage,YearOverYearChange
count,428.000000,428.000000,428.000000
mean,44.830816,44.752738,0.991454
std,10.257549,10.243202,3.683063
min,24.145000,24.344024,-7.976227
25%,40.702615,40.701900,-1.630603
50%,47.132625,47.152741,0.682225
75%,52.144442,52.219243,2.796471
max,61.596261,61.429516,16.343682



Descriptive statistics for the target:


,NextMonthRate
count,428.000000
mean,44.911687
std,10.260986
min,24.145000
25%,40.725309
50%,47.218733
75%,52.191244
max,62.612500



Missing values in features and target:


,Missing Values
CurrentRate,0
Trailing3MonthAverage,0
YearOverYearChange,0
NextMonthRate,0


Duplicate feature-month dates: 0


### Answer these in your report

**Q1.** What real-world quantity are you trying to predict, and why is it continuous rather than categorical?

> *Your answer:* _(We are trying to predict the next month's monthly average Philippine peso per US dollar (PHP/USD) exchange rate. This value represents how many Philippine pesos are needed to purchase one US dollar. It is a continuous numerical variable because the exchange rate can take decimal values, rather than belonging to a limited set of categories. Regression is appropriate for estimating its numerical value. A higher predicted PHP/USD rate suggests possible depreciation of the peso, while a lower predicted rate suggests possible appreciation.)_

**Q2.** Where does your dataset come from? Is it open/public? Provide the source (link or citation).

> *Your answer:* _(The dataset comes from the Bangko Sentral ng Pilipinas (BSP), the Philippines' central bank. We use its official historical workbook titled Daily, Monthly (Average and End-of-Period), and Annual Peso per US Dollar Exchange Rates. It is publicly accessible and downloadable in Microsoft Excel format. This project uses the monthly worksheet's Average column. Source: https://www.bsp.gov.ph/statistics/external/pesodollar.xlsx)_

**Q3.** How many samples (n) and how many features (p) does your dataset have?

> *Your answer:* _(The expected prepared dataset contains 428 samples (n) and 3 independent numerical features (p), assuming all monthly observations in the selected period are present and valid. The features are CurrentRate, Trailing3MonthAverage, and YearOverYearChange. The target variable is NextMonthRate. Use the actual sample count printed by the notebook in the final report.)_

**Q4.** Before doing any analysis, which features do you expect to be most predictive, and why?

> *Your answer:* _(We expect CurrentRate to be especially useful because it represents the latest available exchange-rate level. Trailing3MonthAverage may also be useful because it summarizes the recent exchange-rate level and reduces the effect of a single month's fluctuation. YearOverYearChange may provide information about longer-term movement compared with the same month in the previous year. Their actual predictive usefulness will be evaluated later using regression results.)_

### Save your work for the next notebook

In [5]:
part_a_dataset.to_csv("peso_usd_part_a_dataset.csv", index_label="FeatureMonth")

%store X_raw
%store y
%store feature_names

print("Saved: peso_usd_part_a_dataset.csv")
print("X_raw shape:", X_raw.shape)
print("y shape:", y.shape)
print("feature_names:", feature_names)

Stored 'X_raw' (ndarray)
Stored 'y' (ndarray)
Stored 'feature_names' (list)
Saved: peso_usd_part_a_dataset.csv
X_raw shape: (428, 3)
y shape: (428,)
feature_names: ['CurrentRate', 'Trailing3MonthAverage', 'YearOverYearChange']
